# LangGraph MCP Tutorial - Outlook Email Assistant (OpenAI)

이 튜토리얼에서는 `outlook_mcp_server.py` MCP 서버를 LangGraph와 OpenAI 모델(`gpt-4o`)을 사용하여 Outlook 이메일을 관리하는 에이전트를 구축합니다.

## Outlook MCP 서버가 제공하는 도구

| 도구 | 설명 |
|------|------|
| `list_folders` | Outlook의 모든 메일 폴더 목록을 조회합니다 |
| `list_recent_emails` | 지정된 기간 내의 최근 이메일 목록을 조회합니다 |
| `search_emails` | 키워드로 이메일을 검색합니다 |
| `get_email_by_number` | 특정 번호의 이메일 상세 내용을 조회합니다 |
| `reply_to_email_by_number` | 특정 이메일에 답장합니다 |
| `compose_email` | 새 이메일을 작성하여 발송합니다 |

## 사전 요구사항

- Windows 환경에서 Outlook 데스크톱 앱이 실행 중이어야 합니다
- `OPENAI_API_KEY`가 `.env` 파일에 설정되어 있어야 합니다
- `pywin32` 패키지가 설치되어 있어야 합니다 (`pip install pywin32`)

## 환경 설정

환경 변수를 로드하고 LangSmith 로깅을 설정합니다.

In [1]:
from dotenv import load_dotenv
from langchain_teddynote import logging

# 환경 변수 로드
load_dotenv(override=True)
# LangSmith 프로젝트 이름 설정
logging.langsmith("LangGraph-V1-Tutorial")

LangSmith 추적을 시작합니다.
[프로젝트명]
LangGraph-V1-Tutorial


## 라이브러리 임포트 및 Windows 호환성 설정

MCP 클라이언트, LangGraph 에이전트, 그리고 Windows Jupyter 환경에서의 호환성 패치를 설정합니다.

In [2]:
import nest_asyncio
from typing import List, Dict, Any

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# MCP 클라이언트: 다중 MCP 서버에 연결하여 도구를 가져옵니다
from langchain_mcp_adapters.client import MultiServerMCPClient

# 비동기 호환성을 활성화합니다 (Jupyter 환경에서 필요)
nest_asyncio.apply()

In [7]:
import sys, os, subprocess

# Windows + Jupyter workaround: MCP stdio passes Jupyter's sys.stderr to subprocess.Popen,
# but Jupyter's stderr doesn't support fileno(). Patch the default errlog to os.devnull.
if sys.platform == "win32":
    import mcp.client.stdio as _mcp_stdio

    _devnull_file = open(os.devnull, "w")

    # @asynccontextmanager wraps the original function — patch __wrapped__.__defaults__
    if hasattr(_mcp_stdio.stdio_client, "__wrapped__"):
        _mcp_stdio.stdio_client.__wrapped__.__defaults__ = (_devnull_file,)

    # Also patch the helper that creates the subprocess
    _mcp_stdio._create_platform_compatible_process.__defaults__ = (
        None,
        _devnull_file,
        None,
    )


async def setup_mcp_client(server_configs: dict):
    """MCP 클라이언트를 설정하고 도구를 가져옵니다.

    Args:
        server_configs: 서버 구성 딕셔너리

    Returns:
        tuple: (MCP 클라이언트, 로드된 도구 목록)
    """
    client = MultiServerMCPClient(server_configs)
    tools = await client.get_tools()

    print(f"[MCP] {len(tools)}개의 도구가 로드되었습니다:")
    for tool in tools:
        print(f"  - {tool.name}")

    return client, tools

---

## Part 1: Outlook MCP 서버 연결

Outlook MCP 서버를 stdio 전송 방식으로 연결합니다. 서버는 `server/outlook_mcp_server.py`에 위치하며, Windows COM을 통해 Outlook 데스크톱 앱과 통신합니다.

> **참고**: Outlook 데스크톱 앱이 실행 중이어야 합니다.

In [9]:
# Outlook MCP 서버 구성 (stdio 전송 방식)
server_configs = {
    "outlook-assistant": {
        "command": "uv",
        "args": ["run", "python", "server/outlook_mcp_server.py"],
        "transport": "stdio",
    },
}

# MCP 클라이언트 생성 및 도구 로드
client, tools = await setup_mcp_client(server_configs=server_configs)

  + Exception Group Traceback (most recent call last):
  |   File "d:\2026_Agent\Teddy_langgraph\langgraph-v1-tutorial\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3699, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "C:\Users\tored\AppData\Local\Temp\ipykernel_23804\482820276.py", line 11, in <module>
  |     client, tools = await setup_mcp_client(server_configs=server_configs)
  |                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "C:\Users\tored\AppData\Local\Temp\ipykernel_23804\2851366460.py", line 32, in setup_mcp_client
  |     tools = await client.get_tools()
  |             ^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "d:\2026_Agent\Teddy_langgraph\langgraph-v1-tutorial\.venv\Lib\site-packages\langchain_mcp_adapters\client.py", line 197, in get_tools
  |     tools_list = await asyncio.gather(*load_mcp_tool_tasks)
  |                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "C:\Use

---

## Part 2: OpenAI 에이전트 생성

OpenAI의 `gpt-4o` 모델을 사용하여 Outlook 도구를 활용하는 에이전트를 생성합니다. `create_agent`는 LLM과 도구 목록을 전달하면 추론-행동 루프를 자동으로 구현합니다.

In [ ]:
# OpenAI LLM 설정
llm = init_chat_model("gpt-4o", temperature=0)

# 에이전트 생성: Outlook MCP 도구를 사용하는 에이전트
agent = create_agent(
    llm,
    tools,
    checkpointer=InMemorySaver(),  # 대화 상태를 메모리에 저장
)

In [ ]:
# 스트리밍 헬퍼 함수와 UUID 생성 함수를 import합니다
from langchain_teddynote.messages import astream_graph, random_uuid
from langchain_core.runnables import RunnableConfig

---

## Part 3: 이메일 기능 예제

### 예제 1: 메일 폴더 목록 조회

`list_folders` 도구를 사용하여 Outlook에서 사용 가능한 모든 메일 폴더를 확인합니다.

In [ ]:
# 대화 스레드 ID를 설정합니다
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 메일 폴더 목록 조회
response = await astream_graph(
    agent,
    inputs={"messages": [("human", "Outlook에서 사용 가능한 메일 폴더 목록을 보여주세요.")]},
    config=config,
)

### 예제 2: 최근 이메일 목록 조회

`list_recent_emails` 도구를 사용하여 최근 3일 이내의 이메일 목록을 조회합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 최근 3일 이내 이메일 조회
response = await astream_graph(
    agent,
    inputs={"messages": [("human", "최근 3일 이내에 받은 이메일 목록을 보여주세요.")]},
    config=config,
)

### 예제 3: 이메일 검색

`search_emails` 도구를 사용하여 특정 키워드로 이메일을 검색합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 키워드로 이메일 검색
response = await astream_graph(
    agent,
    inputs={"messages": [("human", "최근 7일 이내 이메일 중에서 '회의' 키워드로 검색해주세요.")]},
    config=config,
)

### 예제 4: 이메일 상세 조회

`get_email_by_number` 도구를 사용하여 이전 검색 결과에서 특정 이메일의 상세 내용을 확인합니다.

> **참고**: 이 기능은 이전에 `list_recent_emails` 또는 `search_emails`를 실행한 후에 사용할 수 있습니다. 같은 `thread_id`를 사용하여 대화 컨텍스트를 유지합니다.

In [ ]:
# 같은 스레드에서 이어서 대화 (이전 검색 결과의 컨텍스트 유지)
response = await astream_graph(
    agent,
    inputs={"messages": [("human", "1번 이메일의 상세 내용을 보여주세요.")]},
    config=config,  # 이전과 같은 config 사용 (같은 thread_id)
)

### 예제 5: 새 이메일 작성

`compose_email` 도구를 사용하여 새 이메일을 작성하고 발송합니다.

> **주의**: 아래 코드를 실행하면 실제로 이메일이 발송됩니다. 수신자 이메일 주소를 적절히 변경해주세요.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 새 이메일 작성
response = await astream_graph(
    agent,
    inputs={"messages": [("human",
        "다음 내용으로 이메일을 작성해주세요:\n"
        "- 수신자: test@example.com\n"
        "- 제목: [테스트] LangGraph MCP 이메일 테스트\n"
        "- 내용: 안녕하세요, 이 이메일은 LangGraph MCP Outlook 튜토리얼에서 자동 발송된 테스트 이메일입니다."
    )]},
    config=config,
)

---

## Part 4: ToolNode를 활용한 커스텀 워크플로우

`ToolNode`를 사용하면 `create_agent` 대신 LangGraph `StateGraph`로 더 세밀한 제어가 가능한 워크플로우를 만들 수 있습니다. 에이전트-도구 루프의 각 단계를 명시적으로 제어할 수 있습니다.

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage
from typing import Annotated, TypedDict


class AgentState(TypedDict):
    """에이전트 상태 정의"""
    messages: Annotated[List[BaseMessage], add_messages]
    context: Dict[str, Any]


async def create_outlook_workflow(server_configs: dict):
    """Outlook MCP 도구를 사용하는 커스텀 워크플로우를 생성합니다.

    Args:
        server_configs: MCP 서버 구성 딕셔너리

    Returns:
        CompiledStateGraph: 컴파일된 워크플로우 그래프
    """
    # MCP 클라이언트 생성 및 도구 로드
    client, tools = await setup_mcp_client(server_configs=server_configs)

    # OpenAI LLM 설정 및 도구 바인딩
    llm = init_chat_model("gpt-4o", temperature=0)
    llm_with_tools = llm.bind_tools(tools)

    # 워크플로우 그래프 생성
    workflow = StateGraph(AgentState)

    async def agent_node(state: AgentState):
        """에이전트 노드: LLM을 호출하여 응답을 생성합니다"""
        response = await llm_with_tools.ainvoke(state["messages"])
        return {"messages": [response]}

    # ToolNode 생성: 도구 호출을 처리합니다
    tool_node = ToolNode(tools)

    # 그래프에 노드 추가
    workflow.add_node("agent", agent_node)
    workflow.add_node("tools", tool_node)

    # 엣지 정의: 시작 -> 에이전트
    workflow.add_edge(START, "agent")

    # 조건부 엣지: 에이전트 -> (도구 or 종료)
    workflow.add_conditional_edges("agent", tools_condition)

    # 도구 -> 에이전트 (도구 실행 후 다시 에이전트로)
    workflow.add_edge("tools", "agent")

    # 그래프 컴파일
    app = workflow.compile(checkpointer=InMemorySaver())

    return app


# 커스텀 워크플로우 생성
outlook_workflow = await create_outlook_workflow(server_configs)

### 워크플로우 그래프 시각화

In [ ]:
from IPython.display import Image, display

# 워크플로우 그래프를 시각화합니다
display(Image(outlook_workflow.get_graph().draw_mermaid_png()))

### 커스텀 워크플로우로 이메일 관리

생성된 커스텀 워크플로우를 사용하여 이메일 목록을 조회하고 요약을 요청합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 커스텀 워크플로우 실행: 최근 이메일 요약 요청
response = await astream_graph(
    outlook_workflow,
    inputs={"messages": [("human", "최근 3일 이내 받은 이메일을 조회하고, 중요한 이메일을 요약해주세요.")]},
    config=config,
)

---

## 정리

이 튜토리얼에서 다룬 내용:

1. **Outlook MCP 서버 연결**: `outlook_mcp_server.py`를 stdio 전송 방식으로 연결
2. **OpenAI 에이전트 생성**: `gpt-4o` 모델과 `create_agent`를 사용한 에이전트 구축
3. **이메일 기능 활용**: 폴더 조회, 이메일 목록, 검색, 상세 조회, 이메일 작성
4. **커스텀 워크플로우**: `StateGraph`와 `ToolNode`를 활용한 세밀한 제어

### 주요 포인트

- MCP 서버는 `@mcp.tool()` 데코레이터로 도구를 정의하고, `MultiServerMCPClient`를 통해 LangGraph에 연결됩니다
- `create_agent`는 간편하게 에이전트를 생성하며, `StateGraph`는 더 세밀한 제어가 필요할 때 사용합니다
- 같은 `thread_id`를 사용하면 대화 컨텍스트가 유지되어 이전 검색 결과를 참조할 수 있습니다